<a href="https://colab.research.google.com/github/abbiddo/MD_payment_data_auto/blob/main/payment_data_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# google 계정 연동

from google.colab import auth
auth.authenticate_user()

In [ ]:
# drive mount

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
# 결제 내역 데이터 불러오기

import pandas as pd
from datetime import datetime

today = datetime.today().date()
today = str(today)
today = today[:4]+'_'+today[5:7]+'_'+today[8:]
print(today)

file_name = "payment_data_" + today + '.csv'
print(file_name)

file_path = "/content/drive/MyDrive/payment/data/" + file_name

df = pd.read_csv(file_path, encoding="utf-8-sig")

#df.head()

2026_05_20
payment_data_2026_05_20.csv


In [ ]:
# Data 정제

print(len(df))

df = df[df['subject'].isin(['수학'])]
print(len(df))

# B2B data
df_b2b = df[df['담당부서'] == 'B2B팀']

# 예외 데이터 제거
df_b2b = df_b2b[df_b2b['개강전환불'] != 1]
df_b2b = df_b2b[df_b2b['계약상태'] != '결제취소']

# 26년 데이터
df_b2b = df_b2b[(df_b2b['계산일'] > '2025-12-26')]

# 가시성 정렬
df_b2b = df_b2b.sort_values(by=["계산일", "선생님", "학생이름"])

11
7
5


In [ ]:
# 상세 상태값 조정

import numpy as np
import pandas as pd

# 날짜형 변환
df_b2b["실제권한종료일"] = pd.to_datetime(df_b2b["실제권한종료일"])
df_b2b["권한시작일"] = pd.to_datetime(df_b2b["권한시작일"])

# 사용일수 계산
cond_day = (df_b2b["실제권한종료일"] - df_b2b["권한시작일"]).dt.days

df_b2b["상태확정"] = df_b2b["계약상태"]

# 관리진행 + 결제금액 0 → 미수
cond1 = (df_b2b["상태확정"] == "관리진행") & (df_b2b["결제금액"] == 0)
df_b2b.loc[cond1, "상태확정"] = "미수"

# 결제-환불 = 0 AND 신규/재연장=0 → 신규환불
cond3 = ((df_b2b["상태확정"] == "환불") & (df_b2b["결제금액"] - df_b2b["환불금액"] == 0) &
         (df_b2b["신규/재연장"] == 0))
df_b2b.loc[cond3, "상태확정"] = "신규환불"

# 환불대기 + 연장아님 + 사용일수<10 → 신규환불
cond4 = ((df_b2b["상태확정"] == "환불대기") &
         (~df_b2b["상품명"].str.startswith("[연장", na=False)) &
         (cond_day < 5))
df_b2b.loc[cond4, "상태확정"] = "신규환불"

# 환불로 시작 → 중도환불
cond5 = df_b2b["상태확정"].str.startswith("환불", na=False)
df_b2b.loc[cond5, "상태확정"] = "중도환불"


df_b2b["실제권한종료일"] = df_b2b["실제권한종료일"].dt.strftime("%Y-%m-%d")
df_b2b["권한시작일"] = df_b2b["권한시작일"].dt.strftime("%Y-%m-%d")

#total_df.head(40)

5


In [ ]:
# 결측치 채우기

import numpy as np

df_b2b = df_b2b.replace([np.inf, -np.inf], np.nan)
df_b2b = df_b2b.fillna("")

In [ ]:
# 코다용 컬럼명 변경 및 시트 저장

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

worksheet = gc.open('학생 관리').sheet1

# setup column
df_coda = pd.DataFrame()

df_coda["상태확정"] = df_b2b["상태확정"]
df_coda["신규0 / 연장1"] = df_b2b["신규/재연장"]
df_coda["계정"] = df_b2b["id"]
df_coda["학생이름"] = df_b2b["학생이름"]
df_coda["권한시작일"] = df_b2b["권한시작일"]
df_coda["실권한종료일"] = df_b2b["실제권한종료일"]
df_coda["계산일"] = df_b2b["계산일"]
df_coda["종료시점 담임쌤"] = df_b2b["선생님"]
df_coda["상품명"] = df_b2b["상품명"]
df_coda["학년"] = df_b2b["학년"]
df_coda["소속팀"] = df_b2b["담당부서"]
df_coda["권한상태_원본"] = df_b2b["계약상태"]

worksheet.clear()
worksheet.update([df_coda.columns.tolist()] + df_coda.values.tolist())

{'spreadsheetId': '1gp43sUHqWp0be7zkvCJFgVJ4FGt273Qlrfcbwulq6Y0',
 'updatedRange': "'코다'!A1:L6",
 'updatedRows': 6,
 'updatedColumns': 12,
 'updatedCells': 72}

# 시트

In [ ]:
# 관리 시트용 컬럼명 변경

df_sheet = pd.DataFrame()

df_sheet["상태확정"] = df_b2b["상태확정"]
df_sheet["학생이름"] = df_b2b["학생이름"]
df_sheet["종료시점 담임쌤"] = df_b2b["선생님"]
df_sheet["권한시작일"] = df_b2b["권한시작일"]
df_sheet["실권한종료일"] = df_b2b["실제권한종료일"]
df_sheet["계산일"] = df_b2b["계산일"]
df_sheet["결제금액"] = df_b2b["결제금액"]
df_sheet["신규0 / 연장1"] = df_b2b["신규/재연장"]
df_sheet["계정"] = df_b2b["id"]
df_sheet["상품명"] = df_b2b["상품명"]
df_sheet["학년"] = df_b2b["학년"]
df_sheet["소속팀"] = df_b2b["담당부서"]
df_sheet["권한상태_원본"] = df_b2b["계약상태"]

In [ ]:
# 주차별 구분

import pandas as pd

# 날짜 정리
df_sheet["계산일"] = pd.to_datetime(df_sheet["계산일"])
df_sheet = df_sheet.reset_index(drop=True)

# 요일 구하기
df_sheet["요일"] = df_sheet["계산일"].dt.weekday

# 해당 주의 '목요일 날짜' 구하기
df_sheet["주목요일"] = df_sheet["계산일"] - pd.to_timedelta(
    df_sheet["요일"] - 3, unit="D"
)

# 그 목요일의 월 추출
df_sheet["기준월"] = df_sheet["주목요일"].dt.month

# 월 안에서 목요일 기준 주차 계산
def calc_week(group):
    first_thu = group["주목요일"].min()
    group["월주차"] = ((group["주목요일"] - first_thu).dt.days // 7) + 1
    return group

df_sheet = df_sheet.groupby("기준월", group_keys=False).apply(calc_week)

# 주차 변경 지점 찾기
df_sheet["이전주차"] = df_sheet["월주차"].shift(1)

cut_index = df_sheet.index[
    df_sheet["월주차"] != df_sheet["이전주차"]
]

# 삽입 행 생성
rows_to_insert = []

for idx in cut_index:
    month_val = int(df_sheet.loc[idx, "기준월"])
    week_val = int(df_sheet.loc[idx, "월주차"])

    df_sheet["기준월"] = df_sheet["기준월"]

    empty_row = pd.Series({col: None for col in df_sheet.columns})
    empty_row["상태확정"] = f"{month_val}월 {week_val}주차"

    rows_to_insert.append((idx, empty_row))

# 삽입
for idx, row in reversed(rows_to_insert):
    df_sheet = pd.concat([
        df_sheet.iloc[:idx],
        pd.DataFrame([row]),
        df_sheet.iloc[idx:]
    ]).reset_index(drop=True)

# 보조 컬럼 제거
df_sheet = df_sheet.drop(columns=[
    "요일", "주목요일", "기준월", "월주차", "이전주차"
])

df_sheet["계산일"] = df_sheet["계산일"].dt.strftime("%Y-%m-%d")

/tmp/ipykernel_6027/3163698537.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sheet = df_sheet.groupby("기준월", group_keys=False).apply(calc_week)
/tmp/ipykernel_6027/3163698537.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sheet = pd.concat([


In [ ]:
import numpy as np

df_sheet = df_sheet.replace([np.inf, -np.inf], np.nan)
df_sheet = df_sheet.fillna("")

/tmp/ipykernel_6027/4234550711.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sheet = df_sheet.replace([np.inf, -np.inf], np.nan)


In [ ]:
def create_personal_df(name):
  df_personal = df_sheet[
    (df_sheet['종료시점 담임쌤'] == name) |
    (df_sheet['종료시점 담임쌤'].isna()) |
    (df_sheet['종료시점 담임쌤'] == '')
  ]
  return df_personal

In [ ]:
import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

worksheet_b2b = gc.open('학생 관리').worksheet('team')
worksheet_b2b_sy = gc.open('학생 관리').worksheet('teacher')

worksheet_b2b.clear()
worksheet_b2b.update([df_sheet.columns.tolist()] + df_sheet.values.tolist())

b2b_sy = create_personal_df("선생님1")

worksheet_b2b_sy.clear()
worksheet_b2b_sy.update([b2b_sy.columns.tolist()] + b2b_sy.values.tolist())

{'spreadsheetId': '1gp43sUHqWp0be7zkvCJFgVJ4FGt273Qlrfcbwulq6Y0',
 'updatedRange': 'teacher!A1:M4',
 'updatedRows': 4,
 'updatedColumns': 13,
 'updatedCells': 52}